In [1]:
import pandas as pd

salario_minimo = pd.read_csv('salario_minimo_mensal_1940_2022.csv', sep=';')
salario_minimo_periodo = salario_minimo.loc[
    (salario_minimo.ano > 1949) &
    (salario_minimo.ano < 2019)
].copy()

edicoes_pato = pd.read_csv('Edicoes_Pato_Donald_Abril.csv', sep=';')
# edicoes_pato["preco_capa"] = edicoes_pato["preco_capa"].ffill()

In [2]:
salario_minimo_periodo[["unidade", "valor"]] = (
    salario_minimo_periodo["salario_minimo"]
    .str.split(" ", n=1, expand=True)
)
salario_minimo_periodo

,mes,ano,salario_minimo,unidade,valor
114,1,1950,"Cr$ 380,00",Cr$,"380,00"
115,2,1950,"Cr$ 380,00",Cr$,"380,00"
116,3,1950,"Cr$ 380,00",Cr$,"380,00"
117,4,1950,"Cr$ 380,00",Cr$,"380,00"
118,5,1950,"Cr$ 380,00",Cr$,"380,00"
...,...,...,...,...,...
937,8,2018,"R$ 954,00",R$,"954,00"
938,9,2018,"R$ 954,00",R$,"954,00"
939,10,2018,"R$ 954,00",R$,"954,00"
940,11,2018,"R$ 954,00",R$,"954,00"


In [3]:
salario_minimo_periodo.rename(
    columns={
        'valor': 'valor_sal',
        'unidade': 'unidade_sal'
    },
    inplace=True
)

In [4]:
edicoes_pato[["unidade", "valor"]] = (
    edicoes_pato["preco_capa"]
    .str.split(" ", n=1, expand=True)
)

edicoes_pato.rename(
    columns={
        'valor': 'valor_gibi',
        'unidade': 'unidade_gibi'
    },
    inplace=True
)

In [5]:
data = pd.merge(edicoes_pato, salario_minimo_periodo, on=['mes', 'ano'])
data

,edicao_numero,mes,ano,preco_capa,url,unidade_gibi,valor_gibi,salario_minimo,unidade_sal,valor_sal
0,1,7,1950,"Cr$ 3,00",http://wwwguiadosquadrinhoscom/edicao/pato-don...,Cr$,"3,00","Cr$ 380,00",Cr$,"380,00"
1,2,8,1950,"Cr$ 3,00",http://wwwguiadosquadrinhoscom/edicao/pato-don...,Cr$,"3,00","Cr$ 380,00",Cr$,"380,00"
2,3,9,1950,"Cr$ 3,00",http://wwwguiadosquadrinhoscom/edicao/pato-don...,Cr$,"3,00","Cr$ 380,00",Cr$,"380,00"
3,4,10,1950,"Cr$ 3,00",http://wwwguiadosquadrinhoscom/edicao/pato-don...,Cr$,"3,00","Cr$ 380,00",Cr$,"380,00"
4,5,11,1950,"Cr$ 3,00",http://wwwguiadosquadrinhoscom/edicao/pato-don...,Cr$,"3,00","Cr$ 380,00",Cr$,"380,00"
...,...,...,...,...,...,...,...,...,...,...
1840,2477,3,2018,"R$ 4,90",http://wwwguiadosquadrinhoscom/edicao/pato-don...,R$,"4,90","R$ 954,00",R$,"954,00"
1841,2478,4,2018,"R$ 4,90",http://wwwguiadosquadrinhoscom/edicao/pato-don...,R$,"4,90","R$ 954,00",R$,"954,00"
1842,2479,5,2018,"R$ 4,90",http://wwwguiadosquadrinhoscom/edicao/pato-don...,R$,"4,90","R$ 954,00",R$,"954,00"
1843,2480,6,2018,"R$ 4,90",http://wwwguiadosquadrinhoscom/edicao/pato-don...,R$,"4,90","R$ 954,00",R$,"954,00"


In [6]:
data["valor_gibi_num"] = pd.to_numeric(
    data["valor_gibi"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce"
)
data["valor_sal_num"] = pd.to_numeric(
    data["valor_sal"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce"
)


data["%_sal_min"] = (data["valor_gibi_num"] / data["valor_sal_num"]) * 100

In [7]:
data["data"] = pd.to_datetime(
    dict(year=data["ano"], month=data["mes"], day=1)
)

In [8]:
data = data.sort_values("data")

In [9]:
data.describe()

,edicao_numero,mes,ano,valor_gibi_num,valor_sal_num,%_sal_min,data
count,1845.000000,1845.000000,1845.000000,1801.000000,1.841000e+03,1797.000000,1845
mean,1284.778862,6.531707,1978.529539,101.475225,3.679207e+04,0.473773,1978-12-27 22:47:24.878048768
min,1.000000,1.000000,1950.000000,0.250000,6.390000e+01,0.004203,1950-07-01 00:00:00
25%,462.000000,4.000000,1960.000000,1.950000,2.688000e+02,0.166667,1960-09-01 00:00:00
50%,1368.000000,7.000000,1978.000000,3.200000,1.200000e+03,0.295341,1978-01-01 00:00:00
75%,2020.000000,10.000000,1993.000000,14.000000,6.000000e+03,0.600384,1993-11-01 00:00:00
max,2481.000000,12.000000,2018.000000,9000.000000,4.639800e+06,5.571429,2018-06-01 00:00:00
std,797.594536,3.438915,18.692825,551.219489,2.544555e+05,0.438601,NaN


In [10]:
import plotly.express as px

fig = px.line(
    data,
    x="data",
    y="%_sal_min",
    title="% do Salário Mínimo ao Longo do Tempo"
)

fig.update_layout(
    xaxis_title="Ano",
    yaxis_title="% do Salário Mínimo"
)

fig.show()

In [11]:
import pandas as pd

# Remove NaN
df2 = data#.dropna(subset=['%_sal_min']).copy()

 # MÁXIMO
i_max = df2['%_sal_min'].idxmax()
max_row = df2.loc[i_max]

# MÍNIMO
i_min = df2['%_sal_min'].idxmin()
min_row = df2.loc[i_min]

# MÉDIA E MEDIANA
media = df2['%_sal_min'].mean()
mediana = df2['%_sal_min'].median()


print("MAIOR CUSTO RELATIVO:")
print(
    f"Data: {max_row['data']:%d/%m/%Y} | "
    f"Edição: {max_row['edicao_numero']} | "
    f"{max_row['%_sal_min']:.2f}% do salário mínimo "
    f"(Gibi={max_row['valor_gibi_num']:.2f}, "
    f"Salário={max_row['valor_sal_num']:.2f})"
)

print("\nMENOR CUSTO RELATIVO:")
print(
    f"Data: {min_row['data']:%d/%m/%Y} | "
    f"Edição: {min_row['edicao_numero']} | "
    f"{min_row['%_sal_min']:.4f}% do salário mínimo "
    f"(Gibi={min_row['valor_gibi_num']:.2f}, "
    f"Salário={min_row['valor_sal_num']:.2f})"
)

print("\nESTATÍSTICAS GERAIS:")
print(f"Média: {media:.3f}% do salário mínimo")
print(f"Mediana: {mediana:.3f}% do salário mínimo")

MAIOR CUSTO RELATIVO:
Data: 01/10/1994 | Edição: 2043 | 5.57% do salário mínimo (Gibi=3.90, Salário=70.00)

MENOR CUSTO RELATIVO:
Data: 01/12/1984 | Edição: 1728 | 0.0042% do salário mínimo (Gibi=7.00, Salário=166560.00)

ESTATÍSTICAS GERAIS:
Média: 0.474% do salário mínimo
Mediana: 0.295% do salário mínimo
